In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency

var_info = {
    'ggt':      ('γ-谷氨酰转移酶', '连续', 'skew'),
    'drinking': ('饮酒状态',        '分类', None),
    'ast':      ('天冬氨酸转氨酶',  '连续', 'skew'),
    'age':      ('年龄',            '连续', 'skew'),
    'hr':       ('心率',            '连续', 'norm'),
    'alt':      ('丙氨酸转氨酶',    '连续', 'skew'),
    'mono':     ('单核细胞百分比',  '连续', 'skew'),
    'gender':   ('性别',            '分类', None),
    'umalb':    ('尿微量白蛋白',    '连续', 'skew'),
    'sua':      ('血尿酸',          '连续', 'norm'),
    'dbil':     ('直接胆红素',      '连续', 'skew'),
    'bun':      ('血尿素氮',        '连续', 'norm'),
    'tp':       ('总蛋白',          '连续', 'norm'),
    'idbil':    ('间接胆红素',      '连续', 'skew'),
    'rbc':      ('红细胞计数',      '连续', 'norm')
}

units = {
    'ggt':   'U/L',
    'ast':   'U/L',
    'age':   '岁',
    'hr':    'bpm',
    'alt':   'U/L',
    'mono':  '%',
    'umalb': 'mg/L',
    'sua':   'μmol/L',
    'dbil':  'μmol/L',
    'bun':   'mmol/L',
    'tp':    'g/L',
    'idbil': 'μmol/L',
    'rbc':   '×10¹²/L'
}

def format_pvalue(p):
    if p < 0.001:
        return "<0.001"
    else:
        return f"{p:.3f}"

def describe_continuous(data, method='norm'):
    data = data.dropna()
    if len(data) == 0:
        return "NA"
    if method == 'norm':
        return f"{data.mean():.2f}±{data.std():.2f}"
    else:
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        return f"{data.median():.2f} ({q1:.2f}-{q3:.2f})"

def describe_categorical(data, col_name):
    data = data.dropna()
    if len(data) == 0:
        return "NA"
    value_counts = data.value_counts()
    total = len(data)
    if col_name == 'gender':
        mapping = {0: '女性', 1: '男性'}
    elif col_name == 'drinking':
        mapping = {1: '从不', 2: '偶尔', 3: '经常'}
    else:
        mapping = {}
    parts = []
    for val, cnt in value_counts.items():
        label = mapping.get(val, str(val))
        parts.append(f"{label}:{cnt} ({cnt/total*100:.1f}%)")
    return ' / '.join(parts)

def calc_smd_continuous(data1, data2):
    x1 = data1.dropna()
    x2 = data2.dropna()
    n1, n2 = len(x1), len(x2)
    if n1 < 2 or n2 < 2:
        return np.nan
    m1, m2 = x1.mean(), x2.mean()
    v1, v2 = x1.var(ddof=1), x2.var(ddof=1)  # 样本方差
    # 合并标准差
    pooled_sd = np.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1 + n2 - 2))
    if pooled_sd == 0:
        return np.nan
    return abs(m2 - m1) / pooled_sd

def calc_smd_categorical(data1, data2):

    x1 = data1.dropna()
    x2 = data2.dropna()
    n1, n2 = len(x1), len(x2)
    if n1 == 0 or n2 == 0:
        return np.nan
    # 获取所有类别
    categories = sorted(set(x1.unique()) | set(x2.unique()))
    if len(categories) <= 1:
        return 0.0
    max_smd = 0.0
    # 对每一个类别（除了最后一个作为参考）计算比例差
    for cat in categories[:-1]:
        p1 = (x1 == cat).mean()
        p2 = (x2 == cat).mean()
        # 合并比例
        p_pool = (n1 * p1 + n2 * p2) / (n1 + n2)
        # 防止分母为0
        denom = np.sqrt(p_pool * (1 - p_pool))
        if denom == 0:
            smd_cat = 0.0
        else:
            smd_cat = abs(p1 - p2) / denom
        if smd_cat > max_smd:
            max_smd = smd_cat
    return max_smd

def generate_table1(df, target_col, var_info, units):
    total_n = len(df)
    met_n = df[target_col].sum()
    non_n = total_n - met_n

    rows = []
    for var, (desc, typ, dist_method) in var_info.items():
        total_data = df[var]
        non_data = df[df[target_col]==0][var]
        met_data = df[df[target_col]==1][var]

        # 变量显示名
        if typ == '分类':
            var_display = var
        else:
            unit = units.get(var, '')
            var_display = f"{var} ({unit})" if unit else var

        if typ == '分类':
            total_str = describe_categorical(total_data, var)
            non_str = describe_categorical(non_data, var)
            met_str = describe_categorical(met_data, var)
            # 卡方检验
            cross_tab = pd.crosstab(df[target_col], df[var])
            if cross_tab.shape[0] == 2 and cross_tab.shape[1] >= 2:
                chi2, p, dof, expected = chi2_contingency(cross_tab)
            else:
                p = np.nan
            p_str = format_pvalue(p) if not np.isnan(p) else "-"
            # 计算SMD（分类）
            smd = calc_smd_categorical(non_data, met_data)
        else:
            total_str = describe_continuous(total_data, method=dist_method)
            non_str = describe_continuous(non_data, method=dist_method)
            met_str = describe_continuous(met_data, method=dist_method)
            if dist_method == 'norm':
                _, p = ttest_ind(non_data.dropna(), met_data.dropna(), equal_var=False)
            else:
                _, p = mannwhitneyu(non_data.dropna(), met_data.dropna(), alternative='two-sided')
            p_str = format_pvalue(p)
            # 计算SMD（连续）
            smd = calc_smd_continuous(non_data, met_data)

        smd_str = f"{smd:.3f}" if not np.isnan(smd) else "-"
        rows.append([var_display, desc, total_str, non_str, met_str, p_str, smd_str])

    columns = ['变量', '含义',
               f'总体 (N={total_n})',
               f'非MetS (N={non_n})',
               f'MetS (N={met_n})',
               'p值', 'SMD']
    table_df = pd.DataFrame(rows, columns=columns)
    return table_df

df = pd.read_csv('../数据/1.标签构造后的数据.csv')

table1 = generate_table1(df, 'ms_cds', var_info, units)
print(table1.to_string(index=False))

# 保存为Excel
table1.to_excel('基线特征表.xlsx', index=False)

In [ ]:
import pandas as pd
df = pd.read_csv('../数据/1.标签构造后的数据.csv')
tezheng = ['ggt', 'drinking', 'ast', 'age', 'hr', 'alt', 'mono', 'gender', 'umalb', 'sua', 'dbil', 'bun', 'tp', 'idbil', 'rbc']
df[tezheng].hist(bins=50)